# ML Meta Tuning

Ноутбук объединяет baseline, logreg и cluster meta-источники и подбирает итоговые параметры моделей через Optuna.
Holdout используется только после выбора конфигураций по CV.

In [ ]:

import warnings
warnings.filterwarnings("ignore")

import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, ParameterGrid
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error

from sklearn.linear_model import LogisticRegression, Lasso, Ridge, ElasticNet, HuberRegressor, BayesianRidge
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans, AgglomerativeClustering, Birch, DBSCAN, OPTICS, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import pairwise_distances

from xgboost import XGBRegressor

try:
    import hdbscan
    HDBSCAN_AVAILABLE = True
except Exception:
    HDBSCAN_AVAILABLE = False

SEED = 2
TARGET_COL = "duration_hours"
DURATION_CAP = 960
HOLDOUT_FRAC = 0.20
CV_SPLITS = 3
LOGREG_OOF_SPLITS = 5
USE_PCA = True
PCA_COMPONENTS = 30

DATA_PATH_ENV = "DATA_FINALL_PATH"
ARTIFACT_DIR_ENV = "META_ARTIFACTS_DIR"


def require_path(env_name: str, label: str) -> Path:
    value = os.environ.get(env_name)
    if not value:
        raise RuntimeError(f"Укажи путь к {label} в переменной окружения {env_name}.")
    path = Path(value).expanduser()
    if not path.exists():
        raise FileNotFoundError(f"Файл для {label} не найден: {path}")
    return path


DATA_PATH = require_path(DATA_PATH_ENV, "data_finall")
ARTIFACT_DIR = Path(os.environ.get(ARTIFACT_DIR_ENV, "./artifacts_meta_all_baseline_bigsearch_optuna")).expanduser()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def reg_metrics(y_true, pred):
    return {
        "MAE": float(mean_absolute_error(y_true, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, pred))),
        "MdAE": float(median_absolute_error(y_true, pred)),
    }

def normalized_entropy(proba):
    proba = np.clip(np.asarray(proba, dtype=float), 1e-12, 1.0)
    if proba.ndim != 2 or proba.shape[1] <= 1:
        return np.zeros(len(proba), dtype=float)
    ent = -(proba * np.log(proba)).sum(axis=1)
    return ent / np.log(proba.shape[1])

def top2_margin(proba):
    proba = np.asarray(proba, dtype=float)
    if proba.ndim != 2 or proba.shape[1] <= 1:
        return np.ones(len(proba), dtype=float)
    desc = np.sort(proba, axis=1)[:, ::-1]
    return desc[:, 0] - desc[:, 1]

def json_ready(obj):
    if isinstance(obj, dict):
        return {str(k): json_ready(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_ready(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return [json_ready(v) for v in obj.tolist()]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return None if np.isnan(obj) else float(obj)
    return obj

def load_split_data():
    df = pd.read_csv(DATA_PATH)
    split = int(len(df) * (1 - HOLDOUT_FRAC))

    train_full = df.iloc[:split].copy().reset_index(drop=True)
    holdout_full = df.iloc[split:].copy().reset_index(drop=True)

    train_filt = train_full[train_full[TARGET_COL] < DURATION_CAP].copy().reset_index(drop=True)
    holdout_typical = holdout_full[holdout_full[TARGET_COL] < DURATION_CAP].copy().reset_index(drop=True)

    return df, train_full, train_filt, holdout_full, holdout_typical

BASELINE_PIPELINES = [
    ("Scaled_Lasso", Pipeline([
        ("Scaler", StandardScaler()),
        ("Lasso", Lasso(random_state=SEED))
    ])),
    ("Scaled_Elastic", Pipeline([
        ("Scaler", StandardScaler()),
        ("Elastic", ElasticNet(random_state=SEED))
    ])),
    ("Scaled_RF_reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("RF", RandomForestRegressor(random_state=SEED))
    ])),
    ("Scaled_ET_reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("ET", ExtraTreesRegressor(random_state=SEED))
    ])),
    ("Scaled_BR_reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("BR", BaggingRegressor(random_state=SEED))
    ])),
    ("Scaled_DT_reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("DT_reg", DecisionTreeRegressor(random_state=SEED))
    ])),
    ("Scaled_Ridge", Pipeline([
        ("Scaler", StandardScaler()),
        ("Ridge", Ridge(random_state=SEED))
    ])),
    ("Scaled_SVR", Pipeline([
        ("Scaler", StandardScaler()),
        ("SVR", SVR(kernel="linear", C=1e2))
    ])),
    ("Scaled_Hub-Reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("Hub-Reg", HuberRegressor())
    ])),
    ("Scaled_BayRidge", Pipeline([
        ("Scaler", StandardScaler()),
        ("BR", BayesianRidge())
    ])),
    ("Scaled_KNN_reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("KNN_reg", KNeighborsRegressor())
    ])),
    ("Scaled_Gboost-Reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("GBoost-Reg", GradientBoostingRegressor(random_state=SEED))
    ])),
    ("Scaled_XGB_reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("XGBR", XGBRegressor(random_state=SEED))
    ])),
    ("Scaled_RFR_PCA", Pipeline([
        ("Scaler", StandardScaler()),
        ("PCA", PCA(n_components=3)),
        ("RF", RandomForestRegressor(random_state=SEED))
    ])),
    ("Scaled_XGBR_PCA", Pipeline([
        ("Scaler", StandardScaler()),
        ("PCA", PCA(n_components=3)),
        ("XGB", XGBRegressor(random_state=SEED))
    ])),
]

MODEL_ORDER = [name for name, _ in BASELINE_PIPELINES]
BASELINE_PIPELINE_LOOKUP = {name: model for name, model in BASELINE_PIPELINES}

def build_regression_model(model_name, extra_params=None):
    if model_name not in BASELINE_PIPELINE_LOOKUP:
        raise ValueError(model_name)

    est = clone(BASELINE_PIPELINE_LOOKUP[model_name])

    if extra_params is not None:
        current = est.get_params()
        safe_updates = {k: v for k, v in extra_params.items() if k in current}
        if safe_updates:
            est.set_params(**safe_updates)

    return est

def compute_baseline_cv_and_holdout(model_names, train_filt, holdout_typical, holdout_full):
    rows = []
    tscv = TimeSeriesSplit(n_splits=CV_SPLITS)

    Xtrain = train_filt.drop(columns=[TARGET_COL]).reset_index(drop=True)
    ytrain = train_filt[TARGET_COL].reset_index(drop=True)

    Xhold_typ = holdout_typical.drop(columns=[TARGET_COL]).reset_index(drop=True)
    yhold_typ = holdout_typical[TARGET_COL].reset_index(drop=True)

    Xhold_full = holdout_full.drop(columns=[TARGET_COL]).reset_index(drop=True)
    yhold_full = holdout_full[TARGET_COL].reset_index(drop=True)

    for model_name in model_names:
        fold_maes = []

        for tr_idx, va_idx in tscv.split(train_filt):
            fold_train = train_filt.iloc[tr_idx].copy().reset_index(drop=True)
            fold_valid = train_filt.iloc[va_idx].copy().reset_index(drop=True)

            Xtr = fold_train.drop(columns=[TARGET_COL]).reset_index(drop=True)
            ytr = fold_train[TARGET_COL].reset_index(drop=True)

            Xva = fold_valid.drop(columns=[TARGET_COL]).reset_index(drop=True)
            yva = fold_valid[TARGET_COL].reset_index(drop=True)

            est = build_regression_model(model_name)
            est.fit(Xtr, ytr)
            pred = est.predict(Xva)
            fold_maes.append(float(mean_absolute_error(yva, pred)))

        est_full = build_regression_model(model_name)
        est_full.fit(Xtrain, ytrain)

        pred_typ = est_full.predict(Xhold_typ)
        pred_full = est_full.predict(Xhold_full)

        m_typ = reg_metrics(yhold_typ, pred_typ)
        m_full = reg_metrics(yhold_full, pred_full)

        rows.append({
            "model": model_name,
            "baseline_cv_MAE": round(float(np.mean(fold_maes)), 4),
            "baseline_cv_std": round(float(np.std(fold_maes)), 4),
            "baseline_holdout_typical_MAE": round(m_typ["MAE"], 4),
            "baseline_holdout_typical_RMSE": round(m_typ["RMSE"], 4),
            "baseline_holdout_typical_MdAE": round(m_typ["MdAE"], 4),
            "baseline_holdout_full_MAE": round(m_full["MAE"], 4),
            "baseline_holdout_full_RMSE": round(m_full["RMSE"], 4),
            "baseline_holdout_full_MdAE": round(m_full["MdAE"], 4),
        })

    return pd.DataFrame(rows).sort_values(["baseline_cv_MAE", "baseline_cv_std"]).reset_index(drop=True)

all_df, train_full, train_filt, holdout_full, holdout_typical = load_split_data()

baseline_cache_path = ARTIFACT_DIR / "baseline_summary_all_baseline_bigsearch_optuna.csv"
if baseline_cache_path.exists():
    baseline_summary = pd.read_csv(baseline_cache_path)
    baseline_summary = baseline_summary.sort_values(["baseline_cv_MAE", "baseline_cv_std"]).reset_index(drop=True)
    print(f"Baseline summary rows: {len(baseline_summary)}")
else:
    baseline_summary = compute_baseline_cv_and_holdout(MODEL_ORDER, train_filt, holdout_typical, holdout_full)
    baseline_summary.to_csv(baseline_cache_path, index=False)
    print(f"Baseline summary rows: {len(baseline_summary)}")

print("rows total =", len(all_df))
print("train rows =", len(train_full))
print("train typical rows =", len(train_filt))
print("holdout rows =", len(holdout_full))
print("holdout typical rows =", len(holdout_typical))
print("models =", len(MODEL_ORDER))
display(baseline_summary)


In [ ]:

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

TOP_K_LOGREG = 5
TOP_K_CLUSTER = 10
OPTUNA_TRIALS_PER_MODEL = 80
OPTUNA_TIMEOUT_PER_MODEL = None
OPTUNA_STARTUP_TRIALS = 15

LOGREG_SCREEN_PATH_ENV = "LOGREG_SCREEN_PATH"
CLUSTER_SCREEN_PATH_ENV = "CLUSTER_SCREEN_PATH"

LOGREG_SCREEN_PATH = require_path(LOGREG_SCREEN_PATH_ENV, "logreg screening")
CLUSTER_SCREEN_PATH = require_path(CLUSTER_SCREEN_PATH_ENV, "cluster screening")

logreg_screen_df = pd.read_csv(LOGREG_SCREEN_PATH)
cluster_screen_df = pd.read_csv(CLUSTER_SCREEN_PATH)

print(f"LogReg screening rows: {len(logreg_screen_df)}")
print(f"Cluster screening rows: {len(cluster_screen_df)}")

BINNING_SCHEMES = {
    "2cl: 0-200 / 200+": {"mode": "fixed", "edges": [0, 200, np.inf], "labels": ["0-200h", "200+h"]},
    "2cl: 0-500 / 500+": {"mode": "fixed", "edges": [0, 500, np.inf], "labels": ["0-500h", "500+h"]},
    "2cl: median(train)": {"mode": "median", "labels": None},
    "3cl: 0-150 / 151-700 / 700+": {"mode": "fixed", "edges": [0, 150, 700, np.inf], "labels": ["0-150h", "151-700h", "700+h"]},
    "3cl: 0-100 / 101-500 / 500+": {"mode": "fixed", "edges": [0, 100, 500, np.inf], "labels": ["0-100h", "101-500h", "500+h"]},
    "3cl: 0-200 / 201-1000 / 1000+": {"mode": "fixed", "edges": [0, 200, 1000, np.inf], "labels": ["0-200h", "201-1000h", "1000+h"]},
    "4cl: 0-100 / 101-500 / 501-1500 / 1500+": {"mode": "fixed", "edges": [0, 100, 500, 1500, np.inf], "labels": ["0-100h", "101-500h", "501-1500h", "1500+h"]},
    "4cl: 0-200 / 201-700 / 701-2000 / 2000+": {"mode": "fixed", "edges": [0, 200, 700, 2000, np.inf], "labels": ["0-200h", "201-700h", "701-2000h", "2000+h"]},
    "4cl: quantile_4(train)": {"mode": "quantile", "q": 4, "labels": None},
    "5cl: 0-200 / 201-1000 / 1001-2000 / 2001-3000 / 3000+": {"mode": "fixed", "edges": [0, 200, 1000, 2000, 3000, np.inf], "labels": ["0-200h", "201-1000h", "1001-2000h", "2001-3000h", "3000+h"]},
    "5cl: 0-50 / 51-200 / 201-500 / 501-1500 / 1500+": {"mode": "fixed", "edges": [0, 50, 200, 500, 1500, np.inf], "labels": ["0-50h", "51-200h", "201-500h", "501-1500h", "1500+h"]},
    "5cl: quantile_5(train)": {"mode": "quantile", "q": 5, "labels": None},
    "6cl: 0-24 / 25-100 / 101-300 / 301-700 / 701-2000 / 2000+": {"mode": "fixed", "edges": [0, 24, 100, 300, 700, 2000, np.inf], "labels": ["0-24h", "25-100h", "101-300h", "301-700h", "701-2000h", "2000+h"]},
}

def fit_binning_on_train(y_train, scheme_cfg):
    mode = scheme_cfg["mode"]

    if mode == "fixed":
        return {"edges": list(scheme_cfg["edges"]), "labels": list(scheme_cfg["labels"])}

    if mode == "median":
        med = float(y_train.median())
        return {
            "edges": [float(y_train.min()) - 1e-9, med, np.inf],
            "labels": [f"<= {med:.0f}", f"> {med:.0f}"],
        }

    if mode == "quantile":
        _, edges = pd.qcut(y_train, q=scheme_cfg["q"], retbins=True, duplicates="drop")
        edges = list(edges)
        edges[0] = float(min(edges[0], y_train.min() - 1e-9))
        edges[-1] = np.inf
        labels = [f"class_{i}" for i in range(len(edges) - 1)]
        return {"edges": edges, "labels": labels}

    raise ValueError(mode)

def apply_binning(y, fitted_binning):
    cls = pd.cut(y, bins=fitted_binning["edges"], labels=False, include_lowest=True, right=True)
    if cls.isna().any():
        raise RuntimeError(f"NaN after binning: {int(cls.isna().sum())}")
    return cls.astype(int).values

def compute_soft_class_weight(y_cls, alpha):
    counts = pd.Series(y_cls).value_counts().sort_index()
    n = len(y_cls)
    k = len(counts)

    weights = {}
    for c, cnt in counts.items():
        balanced = n / (k * cnt)
        weights[int(c)] = float(1.0 + alpha * (balanced - 1.0))
    return weights

def build_logreg_classifier(alpha, y_cls):
    class_weight = compute_soft_class_weight(y_cls, alpha)
    return Pipeline([
        ("sc", StandardScaler()),
        ("m", LogisticRegression(
            random_state=SEED,
            max_iter=4000,
            solver="lbfgs",
            class_weight=class_weight,
        ))
    ])

def aligned_proba(clf, X, n_classes):
    raw = clf.predict_proba(X)
    out = np.zeros((len(X), n_classes), dtype=float)
    seen_classes = clf.named_steps["m"].classes_
    for i, cls in enumerate(seen_classes):
        out[:, int(cls)] = raw[:, i]
    return out

def empirical_prior_from_past(y_past, n_classes):
    if len(y_past) == 0:
        return np.ones(n_classes, dtype=float) / n_classes

    counts = np.bincount(np.asarray(y_past, dtype=int), minlength=n_classes).astype(float)
    if counts.sum() == 0:
        return np.ones(n_classes, dtype=float) / n_classes
    return counts / counts.sum()

def make_time_oof_logreg_features(X_train, y_train_cls, alpha, n_splits=LOGREG_OOF_SPLITS):
    X_train = X_train.reset_index(drop=True)
    y_train_cls = np.asarray(y_train_cls, dtype=int)

    n = len(X_train)
    n_classes = int(np.max(y_train_cls)) + 1

    if n < 4 or n_classes < 2:
        prior = empirical_prior_from_past([], n_classes)
        proba = np.tile(prior, (n, 1))
        pred = np.argmax(proba, axis=1)
        return pred, proba

    n_splits = max(2, min(n_splits, n - 1))
    tscv = TimeSeriesSplit(n_splits=n_splits)

    oof_proba = np.zeros((n, n_classes), dtype=float)
    covered = np.zeros(n, dtype=bool)

    for tr_idx, va_idx in tscv.split(X_train):
        ytr = y_train_cls[tr_idx]
        prior = empirical_prior_from_past(ytr, n_classes)

        if len(np.unique(ytr)) < 2:
            oof_proba[va_idx] = np.tile(prior, (len(va_idx), 1))
            covered[va_idx] = True
            continue

        clf = build_logreg_classifier(alpha, ytr)
        clf.fit(X_train.iloc[tr_idx], ytr)
        oof_proba[va_idx] = aligned_proba(clf, X_train.iloc[va_idx], n_classes)
        covered[va_idx] = True

    missing_idx = np.where(~covered)[0]
    for idx in missing_idx:
        prior = empirical_prior_from_past(y_train_cls[:idx], n_classes)
        oof_proba[idx] = prior

    oof_pred = np.argmax(oof_proba, axis=1)
    return oof_pred, oof_proba

def make_logreg_meta_frame(pred_cls, pred_proba, prefix="logreg"):
    return pd.DataFrame({
        f"{prefix}_pred_class": np.asarray(pred_cls, dtype=int),
        f"{prefix}_max_proba": np.max(pred_proba, axis=1),
        f"{prefix}_entropy": normalized_entropy(pred_proba),
        f"{prefix}_margin_top1_top2": top2_margin(pred_proba),
    })

def build_logreg_augmented(train_df, infer_df, scheme_name, alpha):
    train_df = train_df.reset_index(drop=True)
    infer_df = infer_df.reset_index(drop=True)

    X_train = train_df.drop(columns=[TARGET_COL]).reset_index(drop=True)
    y_train = train_df[TARGET_COL].reset_index(drop=True)
    X_infer = infer_df.drop(columns=[TARGET_COL]).reset_index(drop=True)

    fitted_binning = fit_binning_on_train(y_train, BINNING_SCHEMES[scheme_name])
    y_cls = apply_binning(y_train, fitted_binning)

    n_classes = int(np.max(y_cls)) + 1
    if n_classes < 2:
        return None

    train_pred_cls, train_proba = make_time_oof_logreg_features(X_train, y_cls, alpha)

    if len(np.unique(y_cls)) < 2:
        infer_proba = np.tile(empirical_prior_from_past(y_cls, n_classes), (len(X_infer), 1))
    else:
        clf_full = build_logreg_classifier(alpha, y_cls)
        clf_full.fit(X_train, y_cls)
        infer_proba = aligned_proba(clf_full, X_infer, n_classes)

    infer_pred_cls = np.argmax(infer_proba, axis=1)

    train_meta = make_logreg_meta_frame(train_pred_cls, train_proba, prefix="logreg")
    infer_meta = make_logreg_meta_frame(infer_pred_cls, infer_proba, prefix="logreg")

    X_train_aug = pd.concat([X_train, train_meta], axis=1)
    X_infer_aug = pd.concat([X_infer, infer_meta], axis=1)

    meta_info = {
        "scheme": scheme_name,
        "alpha": float(alpha),
        "n_classes": int(n_classes),
    }
    return X_train_aug, X_infer_aug, meta_info


rng = np.random.default_rng(SEED)
CLUSTER_MIN_VALID_CLUSTERS = 2
CLUSTER_MAX_VALID_CLUSTERS = 60
CLUSTER_MAX_HEAVY_FIT_ROWS = 3000

CLUSTER_EXPERIMENTS = {
    "MiniBatchKMeans": [
        {"n_clusters": k}
        for k in [2, 3, 4, 5, 6, 8, 10, 12]
    ],
    "GaussianMixture": [
        {"n_components": k, "covariance_type": cov}
        for k in [2, 3, 4, 5, 6, 8, 10]
        for cov in ["full", "diag", "tied"]
    ],
    "Ward": [
        {"n_clusters": k}
        for k in [2, 3, 4, 5, 6, 8, 10]
    ],
    "AgglomerativeClustering": [
        {"n_clusters": k, "linkage": linkage}
        for k in [2, 3, 4, 5, 6, 8, 10]
        for linkage in ["complete", "average"]
    ],
    "DBSCAN": [
        {"eps": eps, "min_samples": ms}
        for eps in [0.3, 0.5, 0.8, 1.0, 1.2, 1.5]
        for ms in [5, 10, 20]
    ],
    "OPTICS": [
        {"min_samples": ms, "xi": xi}
        for ms in [5, 10, 20]
        for xi in [0.03, 0.05, 0.10]
    ],
    "BIRCH": [
        {"n_clusters": k, "threshold": th}
        for k in [2, 3, 4, 5, 6, 8, 10]
        for th in [0.2, 0.3, 0.5, 0.7]
    ],
    "SpectralClustering": [
        {"n_clusters": k, "affinity": "nearest_neighbors"}
        for k in [2, 3, 4, 5, 6]
    ],
}

if HDBSCAN_AVAILABLE:
    CLUSTER_EXPERIMENTS["HDBSCAN"] = [
        {"min_cluster_size": mcs, "min_samples": ms}
        for mcs in [5, 10, 20, 30]
        for ms in [None, 5, 10]
    ]

VERBOSE_CLUSTER_RUNS = True
SAVE_CLUSTER_PROGRESS_EVERY = 20

def stable_softmax_negdist(dist):
    dist = np.asarray(dist, dtype=float)
    scale = np.std(dist) if np.std(dist) > 1e-12 else 1.0
    z = -dist / scale
    z = z - np.max(z, axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=1, keepdims=True)

def probs_from_dist(all_d):
    return stable_softmax_negdist(all_d)

def fit_preprocessor(X_train):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X_train)

    pca = None
    Xt = Xs
    if USE_PCA and Xs.shape[1] > PCA_COMPONENTS:
        pca = PCA(n_components=PCA_COMPONENTS, random_state=SEED)
        Xt = pca.fit_transform(Xs)

    return scaler, pca, Xt

def apply_preprocessor(X, scaler, pca):
    Xs = scaler.transform(X)
    return pca.transform(Xs) if pca is not None else Xs

def build_centroids(X, labels):
    labels = np.asarray(labels, dtype=int)
    valid_mask = labels != -1
    valid_labels = np.unique(labels[valid_mask])

    if len(valid_labels) < CLUSTER_MIN_VALID_CLUSTERS:
        return None, None

    label_map = {old: new for new, old in enumerate(sorted(valid_labels))}
    mapped = np.full_like(labels, -1)

    centroids = []
    for old_label in sorted(valid_labels):
        m = labels == old_label
        mapped[m] = label_map[old_label]
        centroids.append(X[m].mean(axis=0))

    centroids = np.vstack(centroids)
    if len(centroids) > CLUSTER_MAX_VALID_CLUSTERS:
        return None, None

    return centroids, mapped

def assign_by_centroids(X, centroids):
    all_d = pairwise_distances(X, centroids, metric="euclidean")
    labels = np.argmin(all_d, axis=1)
    dmin = all_d[np.arange(len(X)), labels]
    return labels, dmin, all_d

def make_clusterer(name, params):
    if name == "MiniBatchKMeans":
        return MiniBatchKMeans(
            n_clusters=params["n_clusters"],
            random_state=SEED,
            n_init=20,
            batch_size=1024
        )
    elif name == "GaussianMixture":
        return GaussianMixture(
            n_components=params["n_components"],
            covariance_type=params["covariance_type"],
            random_state=SEED
        )
    elif name == "Ward":
        return AgglomerativeClustering(
            n_clusters=params["n_clusters"],
            linkage="ward"
        )
    elif name == "AgglomerativeClustering":
        return AgglomerativeClustering(
            n_clusters=params["n_clusters"],
            linkage=params["linkage"]
        )
    elif name == "DBSCAN":
        return DBSCAN(
            eps=params["eps"],
            min_samples=params["min_samples"]
        )
    elif name in {"BIRCH", "Birch"}:
        return Birch(
            n_clusters=params["n_clusters"],
            threshold=params["threshold"]
        )
    elif name == "SpectralClustering":
        return SpectralClustering(
            n_clusters=params["n_clusters"],
            affinity=params["affinity"],
            random_state=SEED,
            assign_labels="kmeans"
        )
    elif name == "OPTICS":
        return OPTICS(
            min_samples=params["min_samples"],
            xi=params["xi"]
        )
    elif name == "HDBSCAN":
        return hdbscan.HDBSCAN(
            min_cluster_size=params["min_cluster_size"],
            min_samples=params["min_samples"],
            prediction_data=True
        )
    else:
        raise ValueError(name)

def fit_clusterer_and_assign(name, params, Xtr, Xte):
    Xfit = Xtr
    fit_mode = "full_train"

    if name in {"SpectralClustering", "HDBSCAN", "OPTICS", "DBSCAN"} and len(Xtr) > CLUSTER_MAX_HEAVY_FIT_ROWS:
        idx = rng.choice(len(Xtr), size=CLUSTER_MAX_HEAVY_FIT_ROWS, replace=False)
        idx = np.sort(idx)
        Xfit = Xtr[idx]
        fit_mode = f"subsample_{CLUSTER_MAX_HEAVY_FIT_ROWS}"

    est = make_clusterer(name, params)

    if name == "GaussianMixture":
        est.fit(Xfit)
        tr_labels = est.predict(Xtr)
        te_labels = est.predict(Xte)
        tr_proba = est.predict_proba(Xtr)
        te_proba = est.predict_proba(Xte)

        if len(np.unique(tr_labels)) < CLUSTER_MIN_VALID_CLUSTERS:
            return None
        return tr_labels, te_labels, tr_proba, te_proba, fit_mode + "_native"

    if name == "HDBSCAN":
        est.fit(Xfit)
        fit_labels = est.labels_
        if len(np.unique(fit_labels[fit_labels != -1])) < CLUSTER_MIN_VALID_CLUSTERS:
            return None

        try:
            te_labels, _ = hdbscan.approximate_predict(est, Xte)
        except Exception:
            return None

        centroids, _ = build_centroids(Xfit, fit_labels)
        if centroids is None:
            return None

        tr_labels, tr_dmin, tr_d = assign_by_centroids(Xtr, centroids)
        te_labels, te_dmin, te_d = assign_by_centroids(Xte, centroids)
        tr_proba = probs_from_dist(tr_d)
        te_proba = probs_from_dist(te_d)
        return tr_labels, te_labels, tr_proba, te_proba, fit_mode + "_native_test"

    if hasattr(est, "fit_predict"):
        fit_labels = est.fit_predict(Xfit)
    else:
        est.fit(Xfit)
        fit_labels = est.labels_ if hasattr(est, "labels_") else est.predict(Xfit)

    if hasattr(est, "predict") and name in {"MiniBatchKMeans", "BIRCH", "Birch"} and len(Xfit) == len(Xtr):
        tr_labels = est.predict(Xtr)
        te_labels = est.predict(Xte)

        if hasattr(est, "cluster_centers_"):
            tr_d = pairwise_distances(Xtr, est.cluster_centers_)
            te_d = pairwise_distances(Xte, est.cluster_centers_)
            tr_proba = probs_from_dist(tr_d)
            te_proba = probs_from_dist(te_d)
        else:
            tr_proba = te_proba = None

        if len(np.unique(tr_labels)) < CLUSTER_MIN_VALID_CLUSTERS:
            return None

        return tr_labels, te_labels, tr_proba, te_proba, fit_mode + "_native"

    centroids, _ = build_centroids(Xfit, fit_labels)
    if centroids is None:
        return None

    tr_labels, tr_dmin, tr_d = assign_by_centroids(Xtr, centroids)
    te_labels, te_dmin, te_d = assign_by_centroids(Xte, centroids)
    tr_proba = probs_from_dist(tr_d)
    te_proba = probs_from_dist(te_d)

    if len(np.unique(tr_labels)) < CLUSTER_MIN_VALID_CLUSTERS:
        return None

    return tr_labels, te_labels, tr_proba, te_proba, fit_mode + "_centroid"

def build_cluster_meta_features(tr_labels, te_labels, tr_proba=None, te_proba=None):
    tr_feat = pd.DataFrame({"cluster_id": tr_labels})
    te_feat = pd.DataFrame({"cluster_id": te_labels})

    tr_sizes = pd.Series(tr_labels).value_counts().to_dict()
    tr_feat["cluster_size_train"] = pd.Series(tr_labels).map(tr_sizes).fillna(0)
    te_feat["cluster_size_train"] = pd.Series(te_labels).map(tr_sizes).fillna(0)

    tr_feat["cluster_is_noise"] = (tr_feat["cluster_id"] == -1).astype(int)
    te_feat["cluster_is_noise"] = (te_feat["cluster_id"] == -1).astype(int)

    tr_ohe = pd.get_dummies(tr_feat["cluster_id"], prefix="cluster")
    te_ohe = pd.get_dummies(te_feat["cluster_id"], prefix="cluster")
    te_ohe = te_ohe.reindex(columns=tr_ohe.columns, fill_value=0)

    tr_feat = pd.concat([tr_feat, tr_ohe], axis=1)
    te_feat = pd.concat([te_feat, te_ohe], axis=1)

    if tr_proba is not None and te_proba is not None:
        tr_feat["cluster_max_proba"] = np.max(tr_proba, axis=1)
        te_feat["cluster_max_proba"] = np.max(te_proba, axis=1)

        tr_feat["cluster_entropy"] = normalized_entropy(tr_proba)
        te_feat["cluster_entropy"] = normalized_entropy(te_proba)

        tr_feat["cluster_margin_top1_top2"] = top2_margin(tr_proba)
        te_feat["cluster_margin_top1_top2"] = top2_margin(te_proba)

        for k in range(tr_proba.shape[1]):
            tr_feat[f"cluster_proba_{k}"] = tr_proba[:, k]
            te_feat[f"cluster_proba_{k}"] = te_proba[:, k]
    else:
        tr_feat["cluster_max_proba"] = 1.0
        te_feat["cluster_max_proba"] = 1.0
        tr_feat["cluster_entropy"] = 0.0
        te_feat["cluster_entropy"] = 0.0
        tr_feat["cluster_margin_top1_top2"] = 1.0
        te_feat["cluster_margin_top1_top2"] = 1.0

    return tr_feat, te_feat

def build_cluster_augmented(train_df, infer_df, cluster_name, cluster_params):
    train_df = train_df.reset_index(drop=True)
    infer_df = infer_df.reset_index(drop=True)

    X_train = train_df.drop(columns=[TARGET_COL]).reset_index(drop=True)
    X_infer = infer_df.drop(columns=[TARGET_COL]).reset_index(drop=True)

    scaler, pca, Xt_train = fit_preprocessor(X_train)
    Xt_infer = apply_preprocessor(X_infer, scaler, pca)

    result = fit_clusterer_and_assign(cluster_name, cluster_params, Xt_train, Xt_infer)
    if result is None:
        return None

    tr_labels, te_labels, tr_proba, te_proba, fit_mode = result
    tr_feat, te_feat = build_cluster_meta_features(tr_labels, te_labels, tr_proba, te_proba)

    X_train_aug = pd.concat([X_train, tr_feat.reset_index(drop=True)], axis=1)
    X_infer_aug = pd.concat([X_infer, te_feat.reset_index(drop=True)], axis=1)

    meta_info = {
        "cluster_name": cluster_name,
        "cluster_params": json_ready(cluster_params),
        "n_clusters_train": int(len(pd.Series(tr_labels).unique())),
        "use_pca": bool(USE_PCA),
        "pca_components": int(PCA_COMPONENTS) if (USE_PCA and X_train.shape[1] > PCA_COMPONENTS) else None,
        "fit_mode": fit_mode,
        "n_meta_features": int(tr_feat.shape[1]),
    }
    return X_train_aug, X_infer_aug, meta_info


def get_logreg_candidates(model_name, top_k=TOP_K_LOGREG):
    sub = (
        logreg_screen_df[logreg_screen_df["model"] == model_name]
        .sort_values(["cv_typical_MAE", "cv_typical_std", "delta_cv_typical"], ascending=[True, True, False])
        .drop_duplicates(subset=["scheme", "alpha"])
        .head(top_k)
    )

    out = []
    for _, row in sub.iterrows():
        out.append({
            "meta_source": "logreg",
            "source_tag": f'logreg::{row["scheme"]}::alpha={float(row["alpha"]):.2f}',
            "config": {"scheme": row["scheme"], "alpha": float(row["alpha"])},
        })
    return out

def get_cluster_candidates(model_name, top_k=TOP_K_CLUSTER):
    sub = (
        cluster_screen_df[cluster_screen_df["model"] == model_name]
        .sort_values(["cv_typical_MAE", "cv_typical_std", "delta_cv_typical"], ascending=[True, True, False])
        .drop_duplicates(subset=["cluster_name", "cluster_params_json"])
        .head(top_k)
    )

    out = []
    for _, row in sub.iterrows():
        out.append({
            "meta_source": "cluster",
            "source_tag": f'cluster::{row["cluster_name"]}::{row["cluster_params_json"]}',
            "config": {
                "cluster_name": row["cluster_name"],
                "cluster_params": json.loads(row["cluster_params_json"]),
            },
        })
    return out

def build_source_augmented_data(meta_source, config, train_df, infer_df):
    if meta_source == "baseline":
        Xtr = train_df.drop(columns=[TARGET_COL]).reset_index(drop=True)
        Xinfer = infer_df.drop(columns=[TARGET_COL]).reset_index(drop=True)
        return Xtr, Xinfer, {"meta_source": "baseline"}

    if meta_source == "logreg":
        return build_logreg_augmented(train_df, infer_df, config["scheme"], config["alpha"])

    if meta_source == "cluster":
        return build_cluster_augmented(train_df, infer_df, config["cluster_name"], config["cluster_params"])

    raise ValueError(meta_source)

def prepare_source_candidates_for_model(model_name):
    out = [{"meta_source": "baseline", "source_tag": "baseline", "config": {}}]
    out.extend(get_logreg_candidates(model_name, TOP_K_LOGREG))
    out.extend(get_cluster_candidates(model_name, TOP_K_CLUSTER))
    return out

def precompute_source_cache_for_model(model_name):
    candidates = prepare_source_candidates_for_model(model_name)
    tscv = TimeSeriesSplit(n_splits=CV_SPLITS)
    cache = {}

    for cand in candidates:
        source_tag = cand["source_tag"]
        meta_source = cand["meta_source"]
        config = cand["config"]

        fold_data = []
        valid = True

        for tr_idx, va_idx in tscv.split(train_filt):
            fold_train = train_filt.iloc[tr_idx].copy().reset_index(drop=True)
            fold_valid = train_filt.iloc[va_idx].copy().reset_index(drop=True)

            try:
                built = build_source_augmented_data(meta_source, config, fold_train, fold_valid)
            except Exception:
                built = None

            if built is None:
                valid = False
                break

            Xtr_aug, Xva_aug, meta_info = built
            ytr = fold_train[TARGET_COL].reset_index(drop=True)
            yva = fold_valid[TARGET_COL].reset_index(drop=True)
            fold_data.append((Xtr_aug, ytr, Xva_aug, yva))

        if not valid:
            continue

        try:
            final_typ = build_source_augmented_data(meta_source, config, train_filt, holdout_typical)
            final_full = build_source_augmented_data(meta_source, config, train_filt, holdout_full)
        except Exception:
            final_typ = None
            final_full = None

        if final_typ is None or final_full is None:
            continue

        cache[source_tag] = {
            "meta_source": meta_source,
            "config": config,
            "fold_data": fold_data,
            "final_typ": final_typ,
            "final_full": final_full,
        }

    return cache

def sample_params_for_model(trial, model_name):
    if model_name == "Scaled_Lasso":
        return {
            "Lasso__alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
        }

    if model_name == "Scaled_Elastic":
        return {
            "Elastic__alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
            "Elastic__l1_ratio": trial.suggest_float("l1_ratio", 0.70, 0.999),
        }

    if model_name == "Scaled_RF_reg":
        return {
            "RF__n_estimators": trial.suggest_int("n_estimators", 200, 1000, step=100),
            "RF__max_depth": trial.suggest_categorical("max_depth", [None, 6, 10, 15, 20, 30]),
            "RF__min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
            "RF__max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5, 0.8, 1.0]),
        }

    if model_name == "Scaled_ET_reg":
        return {
            "ET__n_estimators": trial.suggest_int("n_estimators", 200, 1000, step=100),
            "ET__max_depth": trial.suggest_categorical("max_depth", [None, 6, 10, 15, 20, 30]),
            "ET__min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
            "ET__max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5, 0.8, 1.0]),
        }

    if model_name == "Scaled_BR_reg":
        return {
            "BR__n_estimators": trial.suggest_int("n_estimators", 30, 300, step=10),
            "BR__max_samples": trial.suggest_float("max_samples", 0.5, 1.0),
            "BR__max_features": trial.suggest_float("max_features", 0.5, 1.0),
            "BR__bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
            "BR__bootstrap_features": trial.suggest_categorical("bootstrap_features", [False, True]),
        }

    if model_name == "Scaled_DT_reg":
        return {
            "DT_reg__max_depth": trial.suggest_categorical("max_depth", [None, 4, 6, 8, 10, 15, 20, 30]),
            "DT_reg__min_samples_split": trial.suggest_int("min_samples_split", 2, 30),
            "DT_reg__min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        }

    if model_name == "Scaled_Ridge":
        return {
            "Ridge__alpha": trial.suggest_float("alpha", 1e-4, 1e3, log=True),
        }

    if model_name == "Scaled_SVR":
        return {
            "SVR__C": trial.suggest_float("C", 1.0, 1e3, log=True),
            "SVR__epsilon": trial.suggest_float("epsilon", 1e-3, 5.0, log=True),
        }

    if model_name == "Scaled_Hub-Reg":
        return {
            "Hub-Reg__epsilon": trial.suggest_float("epsilon", 1.1, 3.0),
            "Hub-Reg__alpha": trial.suggest_float("alpha", 1e-6, 1e-1, log=True),
        }

    if model_name == "Scaled_BayRidge":
        return {
            "BR__alpha_1": trial.suggest_float("alpha_1", 1e-8, 1e-1, log=True),
            "BR__alpha_2": trial.suggest_float("alpha_2", 1e-8, 1e-1, log=True),
            "BR__lambda_1": trial.suggest_float("lambda_1", 1e-8, 1e-1, log=True),
            "BR__lambda_2": trial.suggest_float("lambda_2", 1e-8, 1e-1, log=True),
        }

    if model_name == "Scaled_KNN_reg":
        return {
            "KNN_reg__n_neighbors": trial.suggest_int("n_neighbors", 3, 80),
            "KNN_reg__weights": trial.suggest_categorical("weights", ["uniform", "distance"]),
            "KNN_reg__p": trial.suggest_categorical("p", [1, 2]),
        }

    if model_name == "Scaled_Gboost-Reg":
        return {
            "GBoost-Reg__loss": "absolute_error",
            "GBoost-Reg__n_estimators": trial.suggest_int("n_estimators", 80, 600, step=20),
            "GBoost-Reg__learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
            "GBoost-Reg__max_depth": trial.suggest_int("max_depth", 1, 4),
            "GBoost-Reg__min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 25),
            "GBoost-Reg__subsample": trial.suggest_float("subsample", 0.5, 1.0),
        }

    if model_name == "Scaled_XGB_reg":
        return {
            "XGBR__objective": "reg:absoluteerror",
            "XGBR__eval_metric": "mae",
            "XGBR__tree_method": "hist",
            "XGBR__n_estimators": trial.suggest_int("n_estimators", 100, 1400, step=50),
            "XGBR__learning_rate": trial.suggest_float("learning_rate", 0.003, 0.2, log=True),
            "XGBR__max_depth": trial.suggest_int("max_depth", 2, 8),
            "XGBR__min_child_weight": trial.suggest_int("min_child_weight", 1, 15),
            "XGBR__subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "XGBR__colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "XGBR__reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 5.0, log=True),
            "XGBR__reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
            "XGBR__gamma": trial.suggest_float("gamma", 0.0, 5.0),
        }

    if model_name == "Scaled_RFR_PCA":
        return {
            "PCA__n_components": trial.suggest_int("pca_n_components", 2, 8),
            "RF__n_estimators": trial.suggest_int("n_estimators", 200, 1000, step=100),
            "RF__max_depth": trial.suggest_categorical("max_depth", [None, 6, 10, 15, 20, 30]),
            "RF__min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
            "RF__max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5, 0.8, 1.0]),
        }

    if model_name == "Scaled_XGBR_PCA":
        return {
            "PCA__n_components": trial.suggest_int("pca_n_components", 2, 8),
            "XGB__objective": "reg:absoluteerror",
            "XGB__eval_metric": "mae",
            "XGB__tree_method": "hist",
            "XGB__n_estimators": trial.suggest_int("n_estimators", 100, 1400, step=50),
            "XGB__learning_rate": trial.suggest_float("learning_rate", 0.003, 0.2, log=True),
            "XGB__max_depth": trial.suggest_int("max_depth", 2, 8),
            "XGB__min_child_weight": trial.suggest_int("min_child_weight", 1, 15),
            "XGB__subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "XGB__colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "XGB__reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 5.0, log=True),
            "XGB__reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
            "XGB__gamma": trial.suggest_float("gamma", 0.0, 5.0),
        }

    raise ValueError(model_name)


In [ ]:

print("ML meta tuning")
print("Optuna trials per model:", OPTUNA_TRIALS_PER_MODEL)
print("Top-K logreg sources per model:", TOP_K_LOGREG)
print("Top-K cluster sources per model:", TOP_K_CLUSTER)

search_rows = []
trial_log_rows = []
best_rows = []

for model_name in MODEL_ORDER:
    print("\nModel:", model_name)
    
    source_cache = precompute_source_cache_for_model(model_name)
    if len(source_cache) == 0:
        print("Нет доступных источников, модель пропущена.")
        continue

    source_tags = list(source_cache.keys())
    print("Доступно источников:", len(source_tags))

    def objective(trial):
        source_tag = trial.suggest_categorical("source_tag", source_tags)
        extra_params = sample_params_for_model(trial, model_name)
        fold_maes = []

        for fold_id, (Xtr_aug, ytr, Xva_aug, yva) in enumerate(source_cache[source_tag]["fold_data"]):
            est = build_regression_model(model_name, extra_params=extra_params)
            est.fit(Xtr_aug, ytr)
            pred = est.predict(Xva_aug)
            mae = float(mean_absolute_error(yva, pred))
            fold_maes.append(mae)

            trial.report(float(np.mean(fold_maes)), step=fold_id)
            if trial.should_prune():
                raise optuna.TrialPruned()

        return float(np.mean(fold_maes))

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=OPTUNA_STARTUP_TRIALS),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=max(5, OPTUNA_STARTUP_TRIALS // 2), n_warmup_steps=1),
    )

    study.optimize(objective, n_trials=OPTUNA_TRIALS_PER_MODEL, timeout=OPTUNA_TIMEOUT_PER_MODEL, show_progress_bar=False)

    for tr in study.trials:
        row = {
            "model": model_name,
            "trial_number": tr.number,
            "state": str(tr.state),
            "value": None if tr.value is None else float(tr.value),
            "params_json": json.dumps(json_ready(tr.params), ensure_ascii=False, sort_keys=True),
            "source_tag": tr.params.get("source_tag"),
        }
        trial_log_rows.append(row)

    best_trial = study.best_trial
    best_params = dict(best_trial.params)
    best_source_tag = best_params.pop("source_tag")
    best_source = source_cache[best_source_tag]

    best_rows.append({
        "model": model_name,
        "meta_source": best_source["meta_source"],
        "source_tag": best_source_tag,
        "source_config_json": json.dumps(json_ready(best_source["config"]), ensure_ascii=False, sort_keys=True),
        "cv_MAE": round(float(best_trial.value), 4),
        "best_params_json": json.dumps(json_ready(best_params), ensure_ascii=False, sort_keys=True),
        "n_source_candidates": len(source_tags),
        "n_trials": len(study.trials),
    })

    search_rows.append({
        "model": model_name,
        "best_source_tag": best_source_tag,
        "best_cv_MAE": round(float(best_trial.value), 4),
        "best_params_json": json.dumps(json_ready(best_params), ensure_ascii=False, sort_keys=True),
        "n_source_candidates": len(source_tags),
        "n_trials": len(study.trials),
    })

    print(
        f"Лучшее для {model_name}: source={best_source_tag} | "
        f"cv_MAE={best_trial.value:.4f} | trials={len(study.trials)}"
    )

search_df = pd.DataFrame(search_rows).sort_values(["best_cv_MAE"]).reset_index(drop=True)
best_by_model_df = pd.DataFrame(best_rows).sort_values(["cv_MAE"]).reset_index(drop=True)
trial_log_df = pd.DataFrame(trial_log_rows)

display(best_by_model_df)
display(search_df)


In [ ]:

final_rows = []

ytrain = train_filt[TARGET_COL].reset_index(drop=True)
yhold_typ = holdout_typical[TARGET_COL].reset_index(drop=True)
yhold_full = holdout_full[TARGET_COL].reset_index(drop=True)

for _, chosen in best_by_model_df.iterrows():
    model_name = chosen["model"]
    meta_source = chosen["meta_source"]
    source_tag = chosen["source_tag"]
    source_cfg = json.loads(chosen["source_config_json"])
    extra_params = json.loads(chosen["best_params_json"])

    source_cache = precompute_source_cache_for_model(model_name)
    if source_tag not in source_cache:
        continue

    final_typ = source_cache[source_tag]["final_typ"]
    final_full = source_cache[source_tag]["final_full"]

    Xtr_aug, Xhold_typ_aug, meta_info = final_typ
    _, Xhold_full_aug, _ = final_full

    est = build_regression_model(model_name, extra_params=extra_params)
    est.fit(Xtr_aug, ytrain)

    pred_typ = est.predict(Xhold_typ_aug)
    pred_full = est.predict(Xhold_full_aug)

    m_typ = reg_metrics(yhold_typ, pred_typ)
    m_full = reg_metrics(yhold_full, pred_full)

    base_row = baseline_summary[baseline_summary["model"] == model_name].iloc[0]

    final_rows.append({
        "model": model_name,
        "selected_meta_source": meta_source,
        "source_tag": source_tag,
        "cv_MAE": float(chosen["cv_MAE"]),
        "holdout_typical_MAE": round(m_typ["MAE"], 4),
        "holdout_typical_RMSE": round(m_typ["RMSE"], 4),
        "holdout_typical_MdAE": round(m_typ["MdAE"], 4),
        "holdout_full_MAE": round(m_full["MAE"], 4),
        "holdout_full_RMSE": round(m_full["RMSE"], 4),
        "holdout_full_MdAE": round(m_full["MdAE"], 4),
        "baseline_holdout_typical_MAE": float(base_row["baseline_holdout_typical_MAE"]),
        "baseline_holdout_full_MAE": float(base_row["baseline_holdout_full_MAE"]),
        "delta_vs_baseline_typical": round(float(base_row["baseline_holdout_typical_MAE"]) - m_typ["MAE"], 4),
        "delta_vs_baseline_full": round(float(base_row["baseline_holdout_full_MAE"]) - m_full["MAE"], 4),
        "source_config_json": chosen["source_config_json"],
        "best_params_json": chosen["best_params_json"],
    })

final_results_df = pd.DataFrame(final_rows).sort_values(
    ["cv_MAE", "holdout_typical_MAE"]
).reset_index(drop=True)

display(final_results_df)

trial_log_df.to_csv(ARTIFACT_DIR / "05_optuna_trial_log_all_baseline_bigsearch_optuna.csv", index=False)
search_df.to_csv(ARTIFACT_DIR / "05_optuna_search_summary_all_baseline_bigsearch_optuna.csv", index=False)
best_by_model_df.to_csv(ARTIFACT_DIR / "05_optuna_best_by_model_all_baseline_bigsearch_optuna.csv", index=False)
final_results_df.to_csv(ARTIFACT_DIR / "05_optuna_final_holdout_all_baseline_bigsearch_optuna.csv", index=False)

protocol = {
    "notebook": "ML_meta_tuning",
    "selection_rule": "Optuna minimises CV MAE on train_filt; holdout is used only once at the end",
    "external_holdout": "same 80/20 chronological holdout as baseline notebook",
    "train_definition": "train_full filtered to duration_hours < 960",
    "holdout_typical_definition": "holdout_full filtered to duration_hours < 960",
    "holdout_full_definition": "full external holdout",
    "models": MODEL_ORDER,
    "top_k_logreg_candidates_per_model": TOP_K_LOGREG,
    "top_k_cluster_candidates_per_model": TOP_K_CLUSTER,
    "optuna_trials_per_model": OPTUNA_TRIALS_PER_MODEL,
    "screening_sources": {
        "logreg_screen_csv": str(LOGREG_SCREEN_PATH),
        "cluster_screen_csv": str(CLUSTER_SCREEN_PATH),
    },
}
with open(ARTIFACT_DIR / "05_optuna_protocol_all_baseline_bigsearch_optuna.json", "w", encoding="utf-8") as f:
    json.dump(protocol, f, ensure_ascii=False, indent=2)

print("Сохранено:")
for name in [
    "05_optuna_trial_log_all_baseline_bigsearch_optuna.csv",
    "05_optuna_search_summary_all_baseline_bigsearch_optuna.csv",
    "05_optuna_best_by_model_all_baseline_bigsearch_optuna.csv",
    "05_optuna_final_holdout_all_baseline_bigsearch_optuna.csv",
    "05_optuna_protocol_all_baseline_bigsearch_optuna.json",
]:
    print("-", name)

print("\nЭксперимент завершён.")
